## Agentes de IA para analise de dados - parte 1
usando apenas tool definidas como funções

## 1-librarys

In [1]:
import os
from dotenv import load_dotenv
import datetime
import time
import requests
import pandas as pd
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
#from langchain_experimental.agents import create_pandas_dataframe_agent


from langchain_core.globals import set_debug




# # Modelos LLM (Large Language Models)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_groq import ChatGroq


print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))

# 20/08/2026 - 13:51:42


## 2-load envs

In [2]:
# 2. Carregamento das variáveis de ambiente (.env)
ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

carrega_variaveis_ambiente()

llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))
set_debug(False)
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))

✔ Variáveis de ambiente carregadas do arquivo .env
# 20/08/2026 - 13:51:42


In [3]:
dados_1 = pd.DataFrame({
    "produto": ["A", "B", "A", "C", "B"],
    "quantidade": [10, 5, 8, 20, 7],
    "valor": [100, 50, 80, 300, 70]
})

dados_2 = pd.DataFrame({
    "produto": ["A", "B", "A"],
    "quantidade": [1, 5, 7],
    "valor": [100, 330, 1]
})

bases = {
    "dados_1": dados_1,
    "dados_2": dados_2
}
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))

# 20/08/2026 - 13:51:42


## 3. tools, defs e agent

In [4]:
#
@tool
def listar_bases() -> str:
    """
    Retorna os nomes dos conjuntos de dados disponíveis para análise.
    Use esta ferramenta quando o usuário perguntar quais dados,
    bases ou DataFrames estão disponíveis.
    """

    return "\n".join(bases.keys())


@tool
def estrutura_dataframe(nome_dataframe: str) -> str:
    """
    Retorna informações sobre a estrutura de um DataFrame.

    Args:
        nome_dataframe: Nome do DataFrame. Use base.keys()
    """

    if nome_dataframe not in bases:
        return f"DataFrame '{nome_dataframe}' não encontrado."

    df = bases[nome_dataframe]

    return f"""
Número de linhas: {df.shape[0]}
Número de colunas: {df.shape[1]}

Colunas:
{df.dtypes.to_string()}
"""
@tool
def estatisticas_dataframe(nome_dataframe: str) -> str:
    """
    Retorna estatísticas descritivas de um DataFrame.

    Args:
        nome_dataframe: Nome do DataFrame. Use base.keys()
    """

    if nome_dataframe not in bases:
        return f"DataFrame '{nome_dataframe}' não encontrado."

    df = bases[nome_dataframe]

    return df.describe(include="all").to_string()


@tool
def analisar_coluna(nome_dataframe: str,
                    coluna: str, operacao: str) -> str:
    """
    Realiza uma operação estatística em uma coluna numérica.

    Args:
        coluna: Nome da coluna.
        operacao: Operação desejada. Pode ser:
                  soma, media, minimo, maximo ou mediana.
    """
    if nome_dataframe not in bases:
        return f"DataFrame '{nome_dataframe}' não encontrado."

    df = bases[nome_dataframe]

    if coluna not in df.columns:
        return f"A coluna '{coluna}' não existe."

    if not pd.api.types.is_numeric_dtype(df[coluna]):
        return f"A coluna '{coluna}' não é numérica."

    operacoes = {
        "soma": df[coluna].sum,
        "media": df[coluna].mean,
        "minimo": df[coluna].min,
        "maximo": df[coluna].max,
        "mediana": df[coluna].median
    }

    if operacao not in operacoes:
        return f"Operação '{operacao}' não disponível."

    resultado = operacoes[operacao]()

    return f"{operacao} de '{coluna}': {resultado}"

@tool
def agrupar_dataframe(nome_dataframe: str,
                      coluna_grupo: str,
                      coluna_valor: str,
                      operacao: str) -> str:
    """
    Agrupa o DataFrame por uma coluna e aplica uma operação
    sobre outra coluna.

    Args:
        coluna_grupo: Coluna utilizada para agrupamento.
        coluna_valor: Coluna numérica analisada.
        operacao: soma, media, minimo, maximo ou contagem.
    """
    
    if nome_dataframe not in bases:
        return f"DataFrame '{nome_dataframe}' não encontrado."
    df = bases[nome_dataframe]

    if coluna_grupo not in df.columns:
        return f"A coluna '{coluna_grupo}' não existe."

    if coluna_valor not in df.columns:
        return f"A coluna '{coluna_valor}' não existe."

    operacoes = {
        "soma": "sum",
        "media": "mean",
        "minimo": "min",
        "maximo": "max",
        "contagem": "count"
    }

    if operacao not in operacoes:
        return f"Operação '{operacao}' não disponível."

    resultado = (
        df.groupby(coluna_grupo)[coluna_valor]
        .agg(operacoes[operacao])
        .sort_values(ascending=False)
    )

    return resultado.to_string()

ferramentas_estatisticas = [
    listar_bases,
    estrutura_dataframe,
    estatisticas_dataframe,
    analisar_coluna,
    agrupar_dataframe
]  


def agente_estat_zero():
    modelo = llm_gemini
    ferramentas = ferramentas_estatisticas 

    # Memória de curto prazo
    checkpointer = InMemorySaver()

    agente = create_agent(
        model=modelo,
        tools=ferramentas,
        checkpointer=checkpointer,
        system_prompt="""
        Você é um agente especializado em análise de dados.
    Sempre utilize as ferramentas quando a resposta depender dos dados.

    Regras:
    - Não invente resultados.
    - Identifique corretamente qual DataFrame foi mencionado pelo usuário.
    - Utilize exatamente o nome dos dados ao chamar as ferramentas.
    - Quando necessário, consulte a estrutura do DataFrame antes de realizar cálculos.
    - Responda usando o seguinte formato.
    dado x     
    dimensão  : n x n
    informação consultada:
    ...
    """
    )

    return agente


def agente_estat_um():
    modelo = llm_gemini

    prompt = """
    Você é um agente especializado em análise de dados.
    Sempre utilize as ferramentas quando a resposta depender dos dados.

    Regras:
    - Não invente resultados.
    - Realize os cálculos usando o DataFrame disponível.
    - Quando necessário, consulte a estrutura dos dados antes de calcular.
    - Responda em português.

    Formato da resposta:
    dado: dados_cidade
    dimensão: n x m
    informação consultada:
    ...
    """

    agente = create_pandas_dataframe_agent(
        llm=modelo,
        df=dados_cidade,   #< aqui deve ser um df ou lista de df. Não um dicionario
        prefix=prompt,
        verbose=True,
        allow_dangerous_code=True,
        return_intermediate_steps=True
    )

    return agente


def mostrar_ferramentas(resposta):
    """
    exibir ferramentas utilizadas pelo agente
    """
    
    print("\n--- Ferramentas utilizadas ---")

    contador = 0

    for mensagem in resposta["messages"]:

        # Ferramentas solicitadas pelo agente
        if hasattr(mensagem, "tool_calls") and mensagem.tool_calls:

            for chamada in mensagem.tool_calls:
                contador += 1

                print(f"\nFerramenta usada: {chamada['name']}")
                print(f"Entrada enviada: {chamada['args']}")

        # Resultado retornado pela ferramenta
        if mensagem.type == "tool":
            print(f"Resposta da ferramenta: {mensagem.content}")

    print(f"\nTotal de chamadas: {contador}")
    print("-----------------------------\n")    

def perguntar(agente, pergunta, config,verbose=False):
    resposta = agente.invoke(
        {
            "messages": [
                {"role": "user", "content": pergunta}
            ]
        },
        config=config
    )

    
    # 2. Imprima os passos intermediários para verificar
    if verbose:
        mostrar_ferramentas(resposta)

    return resposta["messages"][-1].content[0]['text']

def queryA(agente, pergunta):
    resposta = agente.invoke({
        "input": pergunta
    })

    return resposta["output"]

print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))

# 20/08/2026 - 13:51:42


### Exemplo 1

In [5]:
dados_3 = pd.DataFrame({
    "produto": ["A", "B", "A"],
    "quantidade": [1, 5, 7],
    "valor": [100, 330, 1],
    "idade": [2, 1, 5],
    "estatus": ['novo', 'novo', 'velho']
})

bases["dados_3"] = dados_3


executor_do_agente = agente_estat_zero()
config = {
    "configurable": {
        "thread_id": "conversa_1"
    }
}

pergunta0="Qual o numero total de linhas e colunas dados_1?"
pergunta1="Qual o numero total de linhas e colunas dados_2?"
pergunta2="Qual o numero total de linhas e colunas dados_3?"

pergunta3='Qual produto de dados_2 teve a maior quantidade total?'
pergunta4='Qual produto na base teve a maior quantidade total?'

resposta_0 = perguntar(executor_do_agente,pergunta0,config)
resposta_1 = perguntar(executor_do_agente,pergunta1,config,True)
resposta_2 = perguntar(executor_do_agente,pergunta2,config)
resposta_3 = perguntar(executor_do_agente,pergunta3,config)
resposta_4 = perguntar(executor_do_agente,pergunta4,config)

print(resposta_0)
print('-' * 50) 
print(resposta_1)
print('-' * 50) 
print(resposta_2)
print('-' * 50) 
print(resposta_3)
print('-' * 50) 
print(resposta_4)


--- Ferramentas utilizadas ---

Ferramenta usada: estrutura_dataframe
Entrada enviada: {'nome_dataframe': 'dados_1'}
Resposta da ferramenta: 
Número de linhas: 5
Número de colunas: 3

Colunas:
produto       object
quantidade     int64
valor          int64


Ferramenta usada: estrutura_dataframe
Entrada enviada: {'nome_dataframe': 'dados_2'}
Resposta da ferramenta: 
Número de linhas: 3
Número de colunas: 3

Colunas:
produto       object
quantidade     int64
valor          int64


Total de chamadas: 2
-----------------------------

dado: dados_1
dimensão: 5 x 3
informação consultada:
O DataFrame 'dados_1' possui 5 linhas e 3 colunas.
--------------------------------------------------
dado: dados_2
dimensão: 3 x 3
informação consultada:
O DataFrame 'dados_2' possui 3 linhas e 3 colunas.
--------------------------------------------------
dado: dados_3
dimensão: 3 x 5
informação consultada:
O DataFrame 'dados_3' possui 3 linhas e 5 colunas.
-------------------------------------------------

### Exemplo 2

In [6]:
# Dados extraídos da tabela
wiki_dados = {
    'Capital': [
        'Vitória', 'Florianópolis', 'Curitiba', 'São Luís', 'Palmas',
        'Brasília', 'Goiânia', 'Belo Horizonte', 'Cuiabá', 'São Paulo',
        'Campo Grande', 'Rio de Janeiro', 'Aracaju', 'Boa Vista', 'Teresina',
        'Porto Alegre', 'Recife', 'Fortaleza', 'Natal', 'João Pessoa',
        'Salvador', 'Belém', 'Macapá', 'Rio Branco', 'Manaus',
        'Porto Velho', 'Maceió'
    ],
    'idh2000': [
        0.700, 0.660, 0.655, 0.582, 0.508,
        0.582, 0.591, 0.617, 0.577, 0.614,
        0.548, 0.607, 0.519, 0.546, 0.488,
        0.612, 0.538, 0.534, 0.547, 0.523,
        0.525, 0.504, 0.478, 0.423, 0.443,
        0.469, 0.433
    ],
    'idh2010': [
        0.805, 0.800, 0.768, 0.752, 0.749,
        0.742, 0.739, 0.737, 0.726, 0.725,
        0.724, 0.719, 0.708, 0.708, 0.707,
        0.702, 0.698, 0.695, 0.694, 0.693,
        0.679, 0.673, 0.663, 0.661, 0.658,
        0.638, 0.635
    ]
}

# # Criando o DataFrame
dados_cidade = pd.DataFrame(wiki_dados)
## dicinario do data_frame
bases={'dados_cidade':dados_cidade}

In [7]:
print('-' * 40) 

----------------------------------------


In [8]:
executor_do_agente2 = agente_estat_zero()
config = {
    "configurable": {
        "thread_id": "conversa_2"
    }
}
pergunta0="Quais dados estão disponiveis da base de dados?"
pergunta1="Qual top5 melhor IDH em 2000 e 2010?"
pergunta2="Qual top5 maiores aumento de IDH entre 2000-2010?"
pergunta3="Qual top5 menores aumento de IDH entre 2000-2010?"
pergunta4="Qual top5 maiores aumento de IDH entre 2000-2010? mostre tb os valores percentuais de aumento"


resposta_0 = perguntar(executor_do_agente2,pergunta0,config)
resposta_1 = perguntar(executor_do_agente2,pergunta1,config)
resposta_2 = perguntar(executor_do_agente2,pergunta2,config)
resposta_3 = perguntar(executor_do_agente2,pergunta3,config)
resposta_4 = perguntar(executor_do_agente2,pergunta4,config)

print(resposta_0)
print('-' * 50) 
print(resposta_1)
print('-' * 50) 
print(resposta_2)
print('-' * 50) 
print(resposta_3)
print('-' * 50) 
print(resposta_4)



Os dados disponíveis na base de dados são:

dado: dados_cidade
dimensão: não aplicável (lista de bases)
informação consultada:
- dados_cidade
--------------------------------------------------
dado: dados_cidade
dimensão: 27 x 3
informação consultada:
As 5 capitais com melhor IDH em 2000 e 2010 são:

**Top 5 IDH 2000:**
1. Vitória: 0.700
2. Florianópolis: 0.660
3. Curitiba: 0.655
4. Belo Horizonte: 0.617
5. São Paulo: 0.614

**Top 5 IDH 2010:**
1. Vitória: 0.805
2. Florianópolis: 0.800
3. Curitiba: 0.768
4. São Luís: 0.752
5. Palmas: 0.749
--------------------------------------------------
Para calcular o aumento do IDH entre 2000 e 2010, subtraímos o valor de 2000 do valor de 2010 para cada capital. Abaixo estão as 5 capitais com o maior crescimento absoluto:

dado: dados_cidade
dimensão: 27 x 3
informação consultada:
Cálculo realizado subtraindo `idh2000` de `idh2010`:

1. **São Luís**: 0.170 (0.752 - 0.582)
2. **Palmas**: 0.169 (0.749 - 0.508)
3. **Teresina**: 0.167 (0.707 - 0.488)
